In [1]:
import copy
from pprint import pprint

import numpy as np
import pandas as pd
from parse import parse

In [2]:
from ase.data import chemical_symbols

# Create the dictionary using a dict comprehension
num_to_symbol = {i: symbol for i, symbol in enumerate(chemical_symbols) if i > 0}

# Example lookup
print(num_to_symbol[6])   # Output: 'C'
print(num_to_symbol[12])  # Output: 'Mg'

C
Mg


In [3]:
log_parsing_results = {"initial_TS_energy": None,
                       "initial_TS_geometry": None,
                       "saddle_point_found": False,
                       "reopt_TS_energy": None,
                       "reopt_TS_geometry": None,
                       "reopt_TS_num_steps": None,
                       "reopt_TS_frequencies": None,
                       "reopt_TS_single_neg_freq_confirmed": False,
                       "1st_irc_EQ_energy": None,
                       "1st_irc_EQ_geometry": None,
                       "2nd_irc_EQ_energy": None,
                       "2nd_irc_EQ_geometry": None,
                      }

num_atoms = 12
itr_header_pattern = "# ITR. {iter_num}"
consuming_coordinates = False
start_consuming_coordinates_at = -1 
stop_consuming_coordinates_at = -1 
consuming_energy = False
consume_energy_at = -1
saddle_point_found = False
optimized_structure = False
consuming_freq = False
reopt_TS_frequencies = []

# With saddle point optimization AND IRC, we have three stages of the log:
# Stage 1: re-optimization of candidate TS to true saddle point.
# Stage 2: first EQ-finding by IRC.
# Stage 3: second EQ-finding by IRC.

stage = 1 

with open("scratch/C6H6_saddle_validation/C6H6_irc-val_new_model_example.log", "r") as f: 
    for i, line in enumerate(f):
        if line.startswith("# ITR."):
            iter_parsed = parse(itr_header_pattern, line)
            iter_num = int(iter_parsed["iter_num"])
            if stage == 1 and iter_num == 0:
                consuming_energy = True
                consume_energy_at = i + num_atoms + 2
                consuming_coordinates = True
                start_consuming_coordinates_at = i+1
                stop_consuming_coordinates_at = i+num_atoms
                temp_coords = []
        elif line.startswith("Optimized structure"): # Optimized structure,
            optimized_structure = True
            consuming_energy = True
            consume_energy_at = i + num_atoms + 1
            consuming_coordinates = True
            start_consuming_coordinates_at = i+1
            stop_consuming_coordinates_at = i+num_atoms
            temp_coords = []
            print(stage)
            print(i)
            print(start_consuming_coordinates_at)
            print(consume_energy_at)
            print("")
        elif line.startswith("1st-Order Saddle point was found"):
            saddle_point_found = True
            log_parsing_results["saddle_point_found"] = saddle_point_found
            print(f"Saddle point was found after {iter_num} iterations.\n")
        elif saddle_point_found and line.startswith("FREQFREQFREQ") and len(reopt_TS_frequencies) == 0:
            print("Toggling frequency parsing on.")
            consuming_freq = True
        elif (consuming_freq) and (line.startswith("Freq.  :")):
            temp_freqs = [float(x) for x in line.rstrip().split()[-3:]]
            for x in temp_freqs:
                reopt_TS_frequencies.append(x)
            if len(reopt_TS_frequencies) == 30:
                print("Toggling frequency parsing off.\n")
                consuming_freq = False
                log_parsing_results["reopt_TS_frequencies"] = copy.deepcopy(reopt_TS_frequencies)
                if reopt_TS_frequencies[0] < 0.0 and all([x >= 0.0 for x in reopt_TS_frequencies[1:]]):
                    log_parsing_results["reopt_TS_single_neg_freq_confirmed"] = True
        if consuming_coordinates and start_consuming_coordinates_at <= i:
            temp_coord_row = [line.rstrip().split()[0]] + [float(x) for x in line.rstrip().split()[1:]]
            temp_coords.append(temp_coord_row)
            if stop_consuming_coordinates_at == i:
                consuming_coordinates = False
                if stage == 1 and iter_num == 0:
                    log_parsing_results["initial_TS_geometry"] = copy.deepcopy(temp_coords)
                elif optimized_structure:
                    if stage == 1:
                        log_parsing_results["reopt_TS_geometry"] = copy.deepcopy(temp_coords)
                    elif stage == 2:
                        log_parsing_results["1st_irc_EQ_geometry"] = copy.deepcopy(temp_coords)
                    elif stage == 3:
                        log_parsing_results["2nd_irc_EQ_geometry"] = copy.deepcopy(temp_coords)
        if consuming_energy and consume_energy_at == i:
            if stage == 1 and iter_num == 0:
                log_parsing_results["initial_TS_energy"] = float(line.split()[1])
            elif optimized_structure:
                temp_energy = float(line.rstrip().split()[2])
                if stage == 1:
                    log_parsing_results["reopt_TS_energy"] = temp_energy
                    log_parsing_results["reopt_TS_num_steps"] = iter_num
                elif stage == 2:
                    log_parsing_results["1st_irc_EQ_energy"] = temp_energy
                    #log_parsing_results["1st_irc_EQ_num_steps"] = iter_num # Misleading low number due to sphere optimization steps?
                elif stage == 3:
                    log_parsing_results["2nd_irc_EQ_energy"] = temp_energy
                    #log_parsing_results["2nd_irc_EQ_num_steps"] = iter_num # Misleading low number due to sphere optimization steps?
                optimized_structure = False
                if stage <= 2:
                    stage += 1
            consuming_energy = False
            consume_energy_at = -1 

1
1193
1194
1206

Saddle point was found after 37 iterations.

Toggling frequency parsing on.
Toggling frequency parsing off.

2
2984
2985
2997

3
4354
4355
4367



In [4]:
pprint(log_parsing_results, sort_dicts=False)

{'initial_TS_energy': -231.852862902135,
 'initial_TS_geometry': [['C',
                          -0.138552417236,
                          -1.289524234236,
                          -0.760765290175],
                         ['C', -0.574618715979, 1.148561080488, 0.848903484648],
                         ['C', -0.43944556591, -1.352736986731, 0.505187085052],
                         ['C',
                          0.234280555557,
                          -0.052968814402,
                          -1.404325848362],
                         ['C', 0.479719601914, 0.892268841321, -0.05462727736],
                         ['C', 0.131728475445, -0.05383297278, 1.171503824644],
                         ['H', -1.643814414939, 1.340358162814, 0.700959548486],
                         ['H',
                          1.158765535257,
                          -0.050997884502,
                          -2.030023589872],
                         ['H',
                          -0.949802199629,
 

In [5]:
def assemble_xyz(z: list, pos: np.array) -> str:
    """Assembling atomic numbers and positions into xyz format

    Args:
        z (list): chemical elements
        pos (tensor): 3D coordinates

    Returns:
        str: xyz string
    """
    natoms =len(z)
    xyz = f"{natoms}\n\n"
    for _z, _pos in zip(z, pos): #.numpy()):
        xyz += f"{_z}\t" + "\t".join([str(x) for x in _pos]) + "\n"
    return xyz

In [6]:
reopt_ts_geom = log_parsing_results["reopt_TS_geometry"]
reopt_ts_symbols = [x[0] for x in reopt_ts_geom]
reopt_ts_pos = np.array([x[1:] for x in reopt_ts_geom])
reopt_ts_xyz = assemble_xyz(reopt_ts_symbols, reopt_ts_pos)
print(reopt_ts_xyz)

12

C	-0.161984187435	-1.320134403859	-0.789614269742
C	-0.593820883795	1.148526639703	0.820342385536
C	-0.394875009193	-1.416514051599	0.488412215672
C	0.226026991653	-0.156036085213	-1.48139988784
C	0.487171898592	0.883158531331	0.004370662624
C	0.114918339398	-0.040634860016	1.157491274444
H	-1.547869261586	1.670260987775	0.90913430114
H	1.180389400164	-0.179825901553	-2.034018291582
H	-0.831588586096	-2.199202695191	1.110420803801
H	0.684820566981	-0.196373084331	2.084332490053
H	1.395643597339	1.45731601875	-0.203313422821
H	-0.55883286603	0.349458904207	-2.066158261279



In [7]:
def assemble_com(geom: list) -> str:
    """Assembling atomic numbers and positions into .com format

    Args:
        z (list): chemical elements
        pos (tensor): 3D coordinates

    Returns:
        str: xyz string
    """
    coords_block = "\n".join(["\t".join([str(y) for y in x]) for x in geom])+"\n"
    return f'''# SADDLE/B3LYP/Def2SVPP Int(Grid=FineGrid) EmpiricalDispersion=GD3

0 1
{coords_block}
Options
Saddle+IRC
DownDC = 99999
EigenCheck
GauProc=40
GauMem=200
'''

In [8]:
com_block = assemble_com(log_parsing_results["initial_TS_geometry"])
print(com_block)

# SADDLE/B3LYP/Def2SVPP Int(Grid=FineGrid) EmpiricalDispersion=GD3

0 1
C	-0.138552417236	-1.289524234236	-0.760765290175
C	-0.574618715979	1.148561080488	0.848903484648
C	-0.43944556591	-1.352736986731	0.505187085052
C	0.234280555557	-0.052968814402	-1.404325848362
C	0.479719601914	0.892268841321	-0.05462727736
C	0.131728475445	-0.05383297278	1.171503824644
H	-1.643814414939	1.340358162814	0.700959548486
H	1.158765535257	-0.050997884502	-2.030023589872
H	-0.949802199629	-2.112967528744	1.108241036124
H	0.881048691403	-0.246864701457	1.942075259587
H	1.441659302798	1.397399397068	-0.0545689843
H	-0.580968848681	0.38130564116	-1.972559248471

Options
Saddle+IRC
DownDC = 99999
EigenCheck
GauProc=40
GauMem=200



In [9]:
with open("scratch/C6H6_saddle_validation/recreation-test-C6H6_irc-val_new_model_example.com", "w") as f:
    f.write(com_block)

In [10]:
!cat scratch/C6H6_saddle_validation/C6H6_irc-val_new_model_example.com

# SADDLE/B3LYP/Def2SVPP Int(Grid=FineGrid) EmpiricalDispersion=GD3

0 1
C       -0.1385524172364524     -1.2895242342355837     -0.7607652901753402
C       -0.5746187159788264     1.1485610804884538      0.848903484647897
C       -0.43944556590990613    -1.3527369867314836     0.5051870850523872
C       0.23428055555705848     -0.05296881440221325    -1.4043258483621284
C       0.47971960191411184     0.8922688413213409      -0.054627277359575295
C       0.1317284754451082      -0.05383297277965539    1.1715038246435192
H       -1.6438144149385339     1.3403581628136956      0.7009595484863388
H       1.1587655352566293      -0.050997884501638606   -2.0300235898723886
H       -0.9498021996291618     -2.1129675287441425     1.1082410361243584
H       0.8810486914032343      -0.2468647014568118     1.942075259586514
H       1.441659302798145       1.397399397067585       -0.05456898430019256
H       -0.5809688486814069     0.3813056411604537      -1.9725592484713896
Options
Saddle+IRC
Do

In [11]:
!cat scratch/C6H6_saddle_validation/recreation-test-C6H6_irc-val_new_model_example.com

# SADDLE/B3LYP/Def2SVPP Int(Grid=FineGrid) EmpiricalDispersion=GD3

0 1
C	-0.138552417236	-1.289524234236	-0.760765290175
C	-0.574618715979	1.148561080488	0.848903484648
C	-0.43944556591	-1.352736986731	0.505187085052
C	0.234280555557	-0.052968814402	-1.404325848362
C	0.479719601914	0.892268841321	-0.05462727736
C	0.131728475445	-0.05383297278	1.171503824644
H	-1.643814414939	1.340358162814	0.700959548486
H	1.158765535257	-0.050997884502	-2.030023589872
H	-0.949802199629	-2.112967528744	1.108241036124
H	0.881048691403	-0.246864701457	1.942075259587
H	1.441659302798	1.397399397068	-0.0545689843
H	-0.580968848681	0.38130564116	-1.972559248471

Options
Saddle+IRC
DownDC = 99999
EigenCheck
GauProc=40
GauMem=200


In [12]:
import pymatgen

In [13]:
import os

from ase.io import read, write

generated_val_instances_dir = os.path.abspath("/misc/home/guest50/OAReactDiff/results/sample_random_TS-20260626-C6H6-filtered_new-w_wo-w_wo-w_wo-vs_www_valid/")

In [14]:
from ase.io import read
import io

def read_xyz_with_raw_comments(filename):
    """
    Reads an XYZ trajectory and treats the second line of each frame 
    strictly as a single string comment.
    """
    with open(filename, 'r') as f:
        while True:
            # 1. Read the number of atoms
            line = f.readline()
            if not line: break
            natoms = int(line.strip())
            
            # 2. Grab the entire comment line as a single string
            raw_comment = f.readline().strip()
            
            # 3. Read the coordinate block
            coords_lines = [f.readline() for _ in range(natoms)]
            
            # 4. Use io.StringIO to let ASE parse only the coordinates
            xyz_data = f"{natoms}\n{raw_comment}\n" + "".join(coords_lines)
            atoms = read(io.StringIO(xyz_data), format='xyz')
            
            # 5. Force the comment into the info dict as a single value
            atoms.info["comment"] = raw_comment
            
            yield atoms

In [15]:
for dir_path, subdirs, files in os.walk(generated_val_instances_dir):
    for file in files:
        file_path = os.path.join(generated_val_instances_dir, file)
        temp_atoms = [atoms for atoms in read_xyz_with_raw_comments(file_path)]
        ts_name = file.replace(" ", "_")[:-4]
        print(ts_name)
        instance_to_convert = temp_atoms[-1]
        print(instance_to_convert.info["comment"])
        comment = instance_to_convert.info["comment"]
        model_name = "_".join(comment.split("/")[-2:])[:-6].replace(".", "_")
        print(model_name)
        break

C6H6_5710-TS165_CON(24,23)
Generated/inpainted transition state. RMSD: 0.137938 Å. By model /misc/home/guest50/OAReactDiff/oa_reactdiff/trainer/checkpoint/OAReactDiff-SCAN/C6H6_5710-filtered-LBS_32x8-StepLR-cutoff_21-rep0-SCAN-leftnet8ff8f5a1c6d0/ddpm-epoch=2932-val-totloss=457.68.ckpt.
C6H6_5710-filtered-LBS_32x8-StepLR-cutoff_21-rep0-SCAN-leftnet8ff8f5a1c6d0_ddpm-epoch=2932-val-totloss=457_68


In [16]:
instance_to_convert.positions

array([[ 0.90734333,  0.74778076, -0.43771547],
       [-0.18674241,  1.20447932,  0.228123  ],
       [ 1.09358749, -0.72674185, -0.27921233],
       [-0.32629097, -1.0706174 , -0.05696597],
       [-1.07742652,  0.04024825,  0.6384326 ],
       [ 0.17689156, -0.63706568, -1.44062339],
       [-0.66242656, -2.12616124,  0.02463939],
       [ 2.07138444, -1.18260633, -0.17086043],
       [-1.21774799, -0.02597358,  1.73205655],
       [-0.3588635 ,  2.24287643,  0.48120498],
       [-2.08418864,  0.14091099,  0.18069297],
       [ 1.66447976,  1.39287033, -0.89977189]])

In [17]:
from ase.symbols import Symbols

# Convert a list of atomic numbers instantly
atomic_numbers = [1, 1, 6, 8]
sim_symbols = Symbols(atomic_numbers)

print(list(sim_symbols))  # Output: ['H', 'H', 'C', 'O']

['H', 'H', 'C', 'O']


In [18]:
list(Symbols(instance_to_convert.get_atomic_numbers()))

['C', 'C', 'C', 'C', 'C', 'C', 'H', 'H', 'H', 'H', 'H', 'H']

In [19]:
list(instance_to_convert.symbols)

['C', 'C', 'C', 'C', 'C', 'C', 'H', 'H', 'H', 'H', 'H', 'H']

In [20]:
symbols = list(instance_to_convert.symbols)
positions = instance_to_convert.positions

geom_block = [[sym]+[str(x) for x in row] for sym, row in zip(symbols, positions)]
print(geom_block)

[['C', '0.9073433289857725', '0.7477807628592258', '-0.4377154652483509'], ['C', '-0.18674240877264925', '1.2044793171841524', '0.22812299827611446'], ['C', '1.0935874900732798', '-0.7267418546047891', '-0.27921233311765403'], ['C', '-0.3262909714653181', '-1.0706174030448878', '-0.056965968947611294'], ['C', '-1.0774265156014817', '0.04024824868424036', '0.6384325958877652'], ['C', '0.17689155916024868', '-0.6370656780565247', '-1.44062339072125'], ['H', '-0.6624265551583619', '-2.126161242756669', '0.024639390833622633'], ['H', '2.0713844377370703', '-1.1826063255975472', '-0.1708604305277773'], ['H', '-1.217747987862951', '-0.02597358324332514', '1.7320565481589183'], ['H', '-0.35886350060008537', '2.2428764343197987', '0.481204981065403'], ['H', '-2.0841886369037854', '0.14091099270934415', '0.1806929671274816'], ['H', '1.664479760408261', '1.3928703315469826', '-0.8997718927866617']]


In [21]:
com_block = assemble_com(geom_block)
print(com_block)

# SADDLE/B3LYP/Def2SVPP Int(Grid=FineGrid) EmpiricalDispersion=GD3

0 1
C	0.9073433289857725	0.7477807628592258	-0.4377154652483509
C	-0.18674240877264925	1.2044793171841524	0.22812299827611446
C	1.0935874900732798	-0.7267418546047891	-0.27921233311765403
C	-0.3262909714653181	-1.0706174030448878	-0.056965968947611294
C	-1.0774265156014817	0.04024824868424036	0.6384325958877652
C	0.17689155916024868	-0.6370656780565247	-1.44062339072125
H	-0.6624265551583619	-2.126161242756669	0.024639390833622633
H	2.0713844377370703	-1.1826063255975472	-0.1708604305277773
H	-1.217747987862951	-0.02597358324332514	1.7320565481589183
H	-0.35886350060008537	2.2428764343197987	0.481204981065403
H	-2.0841886369037854	0.14091099270934415	0.1806929671274816
H	1.664479760408261	1.3928703315469826	-0.8997718927866617

Options
Saddle+IRC
DownDC = 99999
EigenCheck
GauProc=40
GauMem=200



In [22]:
files = [f for f in os.listdir(generated_val_instances_dir) if os.path.isfile(os.path.join(generated_val_instances_dir, f))]

for file in files:
    file_path = os.path.join(generated_val_instances_dir, file)
    temp_atoms = [atoms for atoms in read_xyz_with_raw_comments(file_path)]
    ts_name = file.replace(" ", "_").replace("(","_").replace(")","_").replace(",","-")[:-4]
    print(ts_name)
    instance_to_convert = temp_atoms[-1]
    #print(instance_to_convert.info["comment"])
    comment = instance_to_convert.info["comment"]
    model_name = "_".join(comment.split("/")[-2:])[:-6].replace(".", "_")
    new_dir = os.path.join(generated_val_instances_dir, "com-"+model_name)
    os.makedirs(new_dir, exist_ok=True)
    symbols = list(instance_to_convert.symbols)
    positions = instance_to_convert.positions
    geom_block = [[sym]+[str(x) for x in row] for sym, row in zip(symbols, positions)]
    com_block = assemble_com(geom_block)
    with open(os.path.join(new_dir, ts_name+".com"), "w") as f_out:
        f_out.write(com_block)
    print(os.path.join(new_dir, ts_name+".com"))

C6H6_5710-TS165_CON_24-23_
/misc/home/guest50/OAReactDiff/results/sample_random_TS-20260626-C6H6-filtered_new-w_wo-w_wo-w_wo-vs_www_valid/com-C6H6_5710-filtered-LBS_32x8-StepLR-cutoff_21-rep0-SCAN-leftnet8ff8f5a1c6d0_ddpm-epoch=2932-val-totloss=457_68/C6H6_5710-TS165_CON_24-23_.com
C6H6_5710-TS282_CON_29-37_
/misc/home/guest50/OAReactDiff/results/sample_random_TS-20260626-C6H6-filtered_new-w_wo-w_wo-w_wo-vs_www_valid/com-C6H6_5710-filtered-LBS_32x8-StepLR-cutoff_21-rep0-SCAN-leftnet8ff8f5a1c6d0_ddpm-epoch=2932-val-totloss=457_68/C6H6_5710-TS282_CON_29-37_.com
C6H6_5710-TS287_CON_86-123_
/misc/home/guest50/OAReactDiff/results/sample_random_TS-20260626-C6H6-filtered_new-w_wo-w_wo-w_wo-vs_www_valid/com-C6H6_5710-filtered-LBS_32x8-StepLR-cutoff_21-rep0-SCAN-leftnet8ff8f5a1c6d0_ddpm-epoch=2932-val-totloss=457_68/C6H6_5710-TS287_CON_86-123_.com
C6H6_5710-TS1558_CON_464-100_
/misc/home/guest50/OAReactDiff/results/sample_random_TS-20260626-C6H6-filtered_new-w_wo-w_wo-w_wo-vs_www_valid/com-C6H6